# 🚀 Servidor de Modelos Locales con Ollama, Google Drive y Ngrok en Colab

Este entorno de ejecución proporciona infraestructura basada en GPU en la nube para desplegar un servidor de inferencia privado, accesible y persistente.

---

## 🛠️ Arquitectura y Funcionamiento

### 💾 Persistencia mediante Google Drive
Para evitar la pérdida de los pesos del modelo al finalizar la sesión, el script monta una unidad de Google Drive y configura la ruta `ollama_models`. Ollama almacena y lee los archivos directamente de este directorio, eliminando la necesidad de re-descargar el modelo en ejecuciones posteriores.

### ⚙️ Inicialización del Entorno y Ollama
Se comprueba la existencia de las dependencias necesarias en el sistema y se instala el binario de Ollama. Tras establecer las variables de entorno para direccionar el almacenamiento y configurar el host, el servicio se inicia de manera controlada y se valida mediante consultas a su API local.

### 📥 Gestión de Modelos y Alias
El script verifica los modelos ya disponibles en el almacenamiento persistente. En caso de ausencia, realiza la descarga única del modelo seleccionado y genera una copia idéntica bajo el alias `colab-activo`. Esto permite mantener una configuración estática en el código del cliente sin importar qué modelo base esté ejecutando el servidor.

### 🌐 Túnel Reverso con Ngrok
Dado que el entorno de Colab opera detrás de un cortafuegos sin direccionamiento público, se inicia un proceso de Ngrok en segundo plano. Este expone de forma segura el puerto 11434 de Ollama, generando un punto de enlace HTTPS público para las peticiones de inferencia externas.

---

## 📋 Instrucciones Rápidas de Uso

1. Asegúrate de tener activa la **GPU T4** (*Entorno de ejecución -> Cambiar tipo de entorno de ejecución*).
2. Coloca tu Authtoken de [Ngrok](https://dashboard.ngrok.com/) en la variable `NGROK_TOKEN`.
3. Selecciona tu modelo favorito y ejecuta la **Celda 1**.
4. Copia la URL de salida (`OLLAMA_HOST=...`) y configúrala en tu .env.
5. *(Opcional)* Ejecuta la **Celda 2** para ver los logs en vivo del servidor y verificar que las peticiones estén llegando correctamente.

In [1]:
#@title 🚀 1. Arranque Principal: Almacenamiento, Ollama, Modelo y Túnel Ngrok
#@markdown > **Requisito previo:** Asegurate de tener activa la **GPU T4** en *Entorno de ejecución -> Cambiar tipo de entorno de ejecución*.
#@markdown > **Nota sobre Google Drive:** Si activás `USAR_GOOGLE_DRIVE`, los modelos se guardan en tu cuenta personal de Google (`/content/drive/MyDrive/ollama_models`) sin afectar al resto del equipo. Si lo desactivás (`False`), se usa almacenamiento efímero local (`/content/ollama_models`).

import os
import shutil
import subprocess
import time

NGROK_TOKEN = "3L1NqkV6WnU1MhvEAXFl72W0FUF_5hZHzYMjGZr544S4K7JdQ" #@param {type:"string"}
MODELO = "qwen2.5vl:7b" #@param ["qwen2.5vl:3b", "qwen2.5vl:7b", "qwen2.5vl:32b", "gemma3:4b", "gemma3:12b", "gemma3:27b", "mistral-small3.1:24b", "minicpm-v:8b"] {allow-input: true}
USAR_GOOGLE_DRIVE = True #@param {type:"boolean"}

NGROK_TOKEN = NGROK_TOKEN.strip()
MODELO = MODELO.strip()

if not NGROK_TOKEN or NGROK_TOKEN == "TU_AUTHTOKEN_AQUI":
    raise ValueError("⚠️ Debes ingresar tu authtoken de Ngrok en NGROK_TOKEN antes de ejecutar la celda.")

# 1. Configurar almacenamiento de modelos (Google Drive personal o efímero local)
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OLLAMA_MODELS_DIR = "/content/drive/MyDrive/ollama_models"
    modo_almacenamiento = "Google Drive personal (/content/drive/MyDrive/ollama_models)"
else:
    OLLAMA_MODELS_DIR = "/content/ollama_models"
    modo_almacenamiento = "Local efímero (/content/ollama_models)"

os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
print(f"📂 Almacenamiento configurado: {modo_almacenamiento}")

# 2. Instalar dependencias (Ollama y pyngrok) solo si aún no están instaladas
if shutil.which("ollama") is None:
    print("📦 Instalando dependencias del sistema y Ollama...")
    subprocess.run(
        "sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils",
        shell=True,
        check=True,
    )
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    # Detener cualquier instancia por defecto para asegurar que tome OLLAMA_MODELS
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    time.sleep(1)

try:
    import requests
    from pyngrok import conf, installer, ngrok
except ImportError:
    print("📦 Instalando pyngrok y requests...")
    subprocess.run(["pip", "install", "-q", "pyngrok", "requests"], check=True)
    import requests
    from pyngrok import conf, installer, ngrok

# 3. Iniciar o reiniciar Ollama asegurando la ruta OLLAMA_MODELS seleccionada
STATE_FILE = "/content/.ollama_models_active_path"
ruta_previa = open(STATE_FILE).read().strip() if os.path.exists(STATE_FILE) else None

ollama_listo = False
if ruta_previa == OLLAMA_MODELS_DIR:
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
            ollama_listo = True
            print("✅ Servicio de Ollama ya en ejecución con el almacenamiento seleccionado.")
    except Exception:
        pass

if not ollama_listo:
    subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    time.sleep(1)
    print(f"⚙️ Iniciando servidor Ollama (OLLAMA_MODELS={OLLAMA_MODELS_DIR})...")
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "0.0.0.0:11434"
    env["OLLAMA_ORIGINS"] = "*"
    env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
    log_file = open("/content/ollama.log", "a")
    subprocess.Popen(
        ["ollama", "serve"],
        stdin=subprocess.DEVNULL,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
    )
    for _ in range(20):
        time.sleep(1)
        try:
            if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
                ollama_listo = True
                with open(STATE_FILE, "w") as f:
                    f.write(OLLAMA_MODELS_DIR)
                print("✅ Servicio de Ollama iniciado correctamente.")
                break
        except Exception:
            pass
    if not ollama_listo:
        raise RuntimeError("❌ No se pudo iniciar ollama serve. Revisá /content/ollama.log")

# 4. Comprobación inteligente anti-descargas consultando http://127.0.0.1:11434/api/tags
resp_tags = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
resp_tags.raise_for_status()
modelos_instalados = [m.get("name", "") for m in resp_tags.json().get("models", [])]
modelo_normalizado = MODELO if ":" in MODELO else f"{MODELO}:latest"

if MODELO in modelos_instalados or modelo_normalizado in modelos_instalados:
    print(f"✅ El modelo '{MODELO}' ya existe en '{OLLAMA_MODELS_DIR}'. Omitiendo descarga.")
else:
    print(f"📥 El modelo '{MODELO}' no se encontró en '{OLLAMA_MODELS_DIR}'. Descargando por única vez...")
    subprocess.run(["ollama", "pull", MODELO], check=True)

# 5. Registrar alias 'colab-activo' para compatibilidad automática con el backend local
subprocess.run(["ollama", "cp", MODELO, "colab-activo"], check=True)

# 6. Levantar túnel Ngrok en el puerto 11434
public_url = None
try:
    tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json().get("tunnels", [])
    if tunnels:
        public_url = tunnels[0]["public_url"]
except Exception:
    pass

if not public_url:
    ngrok_bin = conf.get_default().ngrok_path
    if not os.path.exists(ngrok_bin):
        installer.install_ngrok(ngrok_bin)
    ngrok.set_auth_token(NGROK_TOKEN)
    ngrok_log = open("/content/ngrok.log", "a")
    subprocess.Popen(
        [ngrok_bin, "http", "11434", "--log=stdout"],
        stdin=subprocess.DEVNULL,
        stdout=ngrok_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(15):
        time.sleep(1)
        try:
            tunnels = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2).json().get("tunnels", [])
            if tunnels:
                public_url = tunnels[0]["public_url"]
                break
        except Exception:
            pass

if not public_url:
    raise RuntimeError("❌ No se pudo iniciar el túnel de Ngrok. Revisá /content/ngrok.log")

print("\n" + "=" * 70)
print("🚀 ¡OLLAMA ESTÁ CORRIENDO EN GOOGLE COLAB!")
print(f"📂 Almacenamiento: {modo_almacenamiento}")
print(f"🎯 Modelo activo:  {MODELO} (alias: colab-activo)")
print("-" * 70)
print("Copia y pega esta línea en tu archivo .env local:")
print(f"OLLAMA_HOST={public_url}")
print("=" * 70)

Mounted at /content/drive
📂 Almacenamiento configurado: Google Drive personal (/content/drive/MyDrive/ollama_models)
📦 Instalando dependencias del sistema y Ollama...
📦 Instalando pyngrok y requests...
⚙️ Iniciando servidor Ollama (OLLAMA_MODELS=/content/drive/MyDrive/ollama_models)...
✅ Servicio de Ollama iniciado correctamente.
✅ El modelo 'qwen2.5vl:7b' ya existe en '/content/drive/MyDrive/ollama_models'. Omitiendo descarga.

🚀 ¡OLLAMA ESTÁ CORRIENDO EN GOOGLE COLAB!
📂 Almacenamiento: Google Drive personal (/content/drive/MyDrive/ollama_models)
🎯 Modelo activo:  qwen2.5vl:7b (alias: colab-activo)
----------------------------------------------------------------------
Copia y pega esta línea en tu archivo .env local:
OLLAMA_HOST=https://unbridle-dash-yanking.ngrok-free.dev


In [ ]:
#@title 📜 2. Mantenimiento de sesión y logs de Ollama en vivo (Opcional)
#@markdown Ejecutá esta celda para mantener la sesión de Colab activa y observar en tiempo real las peticiones del backend local.
!tail -f /content/ollama.log

time=2026-10-23T23:17:54.037Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-10-23T23:17:54.037Z level=INFO source=routes.go:2563 msg="vram-based default context" total_vram="14.6 GiB" default_num_ctx=4096
[GIN] 2026/10/23 - 23:17:54 | 500 |  23.04ms |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:17:54 | 500 |   27.6ms |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:18:00 | 500 |    3.15s |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:18:00 | 500 |    6.16s |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:18:00 | 200 | 181.64ms |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:18:00 | 200 |  62.65ms |       127.0.0.1 | GET      "/api/tags"
[GIN] 2026/10/23 - 23:18:00 | 200 | 27.362µs |       127.0.0.1 | HEAD     "/"